In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import MusicDBIO
from utils import MusicDBPermDir
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb


# DB-Specific

In [3]:
from lib import metalarchives
mio   = metalarchives.MusicDBIO(verbose=False, mkDirs=False)
apiio = metalarchives.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant MetalArchives DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/MetalArchives


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Master Search:       {0}".format(len(localArtists.get())))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

MetalArchives Search Results
   Local Master Search:       9162
   Global Master Search:      20402
   Global Master Search Data: 0
   Search Artists:            20402
   Errors:                    166
   Known Summary IDs:         8672


# Search For New Artists

In [ ]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

In [ ]:
useStarterData = True
useMasterData  = False

if useStarterData is True:
    starterData = io.get("starter.p")
    artistNames = Series({v["Ref"].split("/")[-1]: v["Name"] for k,v in starterData.items()})
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import MusicDBIO
    pdbio = MusicDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    

#   Artist Names To Get:  36777
#   Artist Names To Get:  31070
#   Artist Names To Get:  22044


## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "5:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchResults(artistName=artistName)
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        apiio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    apiio.sleep(3.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

if True:
    ts.stop()
    print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
    masterArtists.save(data=masterArtistsDict)
    masterArtistsData.save(data=masterArtistsDataDict)
    if len(searchedForErrors) > 0:
        errors.save(data=searchedForErrors)

## Save Results

In [ ]:
df = masterArtistsData.get()
if isinstance(df,dict):
    print("Found {0} New Artists".format(len(df)))
    searchDF = metalarchives.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = searchDF.append(Series(df))
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    artists = {}
    for artistName,artistResults in searchDF.iteritems():
        if artistResults is not None:
            for item in artistResults:
                artists[item['id']] = item['name']
    print("Found {0} Unique Artists".format(len(artists)))
    s = Series(artists)
    print("  ==> {0} Old Artists".format(len(s[s.index.isin(knownArtists.index)])))
    print("  ==> {0} New Artists".format(len(s[~s.index.isin(knownArtists.index)])))
    print("Saving Data")
    metalarchives.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [8]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

## Find Artists To Download

In [9]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

#   Artist IDs To Get:  45016

MetalArchives Search Results
   Available IDs:      54178
   Known Artist IDs:   9162
   Artist IDs To Get:  45016


## Download The Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "12:05pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(6.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

Starting Process [Getting MetalArchives Artist Data]    ==> Time Is 2022-03-22 10:38:34
========================= termTime(day=today,time=9:00pm) =========================
   ====> Terminate Time Set To 2022-03-22 21:00:00 <====
   ====> Will Terminate Process 10 Hours and 21 Minutes From Now
Searching For Scathing Nocturnal (bands/_/3540422683)           True
Searching For Forsaken Few (bands/_/3540405182)                 True
Searching For Suffer (bands/_/123305)                           True
Searching For Cruz Negra (bands/_/48417)                        True
Searching For Mar de Dolor (bands/_/41205)                      True
5/?        : Process [Getting MetalArchives Artist Data] Has Run For 59 Seconds.
Saving 9167 MetalArchives Searched For Artist (Info) IDs
Searching For Vatican Plague (bands/_/28711)                    True
Searching For Daylight Misery (bands/_/3540289820)              True
Searching For Lost Serenity (bands/_/39157)                     True
Searching For Ka

Searching For No Mercy (bands/_/104846)                         True
Searching For Vanhelgd (bands/_/3540269135)                     True
Searching For Black Temple Below (bands/_/3540336190)           True
Searching For Enter My Silence (bands/_/3486)                   True
Searching For Cerberus Attack (bands/_/3540367915)              True
55/?       : Process [Getting MetalArchives Artist Data] Has Run For 11 Minutes and 31 Seconds.
Saving 9217 MetalArchives Searched For Artist (Info) IDs
Searching For Silence Thereafter (bands/_/3540456173)           True
Searching For The Black Spade (bands/_/3540445808)              True
Searching For Executioner (bands/_/3540338107)                  True
Searching For Giftdwarf (bands/_/3540323968)                    True
Searching For Necrorite (bands/_/3540369218)                    True
60/?       : Process [Getting MetalArchives Artist Data] Has Run For 12 Minutes and 41 Seconds.
Saving 9222 MetalArchives Searched For Artist (Info) IDs
Sear

Searching For Diabolical Mental State (bands/_/3540400061)      True
Searching For Absence of Dawn (bands/_/77552)                   True
Searching For Vesna (bands/_/3540280226)                        True
Searching For Turbo Mass (bands/_/3540328904)                   True
Searching For Dead Christ Cult (bands/_/46638)                  True
105/?      : Process [Getting MetalArchives Artist Data] Has Run For 22 Minutes and 53 Seconds.
Saving 9267 MetalArchives Searched For Artist (Info) IDs
Searching For Blind Hatred (bands/_/55108)                      True
Searching For Winterhymn (bands/_/3540338844)                   True
Searching For Funeral Cult (bands/_/37809)                      True
Searching For The Bloody Crimson Void (bands/_/3540478996)      True
Searching For Goatvöid Holokäust (bands/_/3540461532)           True
110/?      : Process [Getting MetalArchives Artist Data] Has Run For 23 Minutes and 55 Seconds.
Saving 9272 MetalArchives Searched For Artist (Info) IDs
Sear

Searching For Chalice of Throe (bands/_/3540460117)             True
Searching For KOB (bands/_/19004)                               True
Searching For Athrox (bands/_/37308)                            True
155/?      : Process [Getting MetalArchives Artist Data] Has Run For 33 Minutes and 24 Seconds.
Saving 9317 MetalArchives Searched For Artist (Info) IDs
Searching For Lord Volture (bands/_/3540308198)                 True
Searching For Obssesion (bands/_/3540423736)                    True
Searching For Black Reign (bands/_/17161)                       True
Searching For Sole Method (bands/_/116366)                      True
Searching For Scythrow (bands/_/3540490581)                     True
160/?      : Process [Getting MetalArchives Artist Data] Has Run For 34 Minutes and 25 Seconds.
Saving 9322 MetalArchives Searched For Artist (Info) IDs
Searching For Mortillery (bands/_/3540308182)                   True
Searching For Sloven (bands/_/3540427709)                       True
Sear

Searching For Cold Void (bands/_/95291)                         True
205/?      : Process [Getting MetalArchives Artist Data] Has Run For 44 Minutes and 10 Seconds.
Saving 9367 MetalArchives Searched For Artist (Info) IDs
Searching For Hatred (bands/_/7124)                             True
Searching For Withering Earth (bands/_/3540407604)              True
Searching For Black Autumn (bands/_/4511)                       True
Searching For Umbilical Asphyxia (bands/_/3540446551)           True
Searching For Slowbleed (bands/_/3540500704)                    True
210/?      : Process [Getting MetalArchives Artist Data] Has Run For 45 Minutes and 13 Seconds.
Saving 9372 MetalArchives Searched For Artist (Info) IDs
Searching For Cain's Offering (bands/_/3540286624)              True
Searching For Grimfist (bands/_/11345)                          True
Searching For Tuhonsiemen (bands/_/3540378847)                  True
Searching For Necroist (bands/_/3540353651)                     True
Sear

Searching For Red Circuit (bands/_/60204)                       True
Searching For Moongates Guardian (bands/_/3540377590)           True
Searching For Ramses (bands/_/39889)                            True
Searching For Skies Burn Black (bands/_/58065)                  True
Searching For Avoid (bands/_/64995)                             True
260/?      : Process [Getting MetalArchives Artist Data] Has Run For 55 Minutes and 51 Seconds.
Saving 9422 MetalArchives Searched For Artist (Info) IDs
Searching For Sorrow of Rain (bands/_/3540351214)               True
Searching For Unchained Beast (bands/_/3540375354)              True
Searching For At Devil Dirt (bands/_/3540341805)                True
Searching For Throne of Nails (bands/_/6087)                    True
Searching For Agon Origin (bands/_/23624)                       True
265/?      : Process [Getting MetalArchives Artist Data] Has Run For 56 Minutes and 56 Seconds.
Saving 9427 MetalArchives Searched For Artist (Info) IDs
Sear

Searching For Imprudent Killer (bands/_/3540391377)             True
Searching For Winter Magik (bands/_/32485)                      True
Searching For Realized (bands/_/3540312932)                     True
310/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 6 Minutes.
Saving 9472 MetalArchives Searched For Artist (Info) IDs
Searching For Deity of Carnification (bands/_/7724)             True
Searching For Witchlust (bands/_/29157)                         True
Searching For Christ of Oblivion (bands/_/3540439403)           True
Searching For Mythic (bands/_/3117)                             True
Searching For Schattenfang (bands/_/3540348818)                 True
315/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 7 Minutes.
Saving 9477 MetalArchives Searched For Artist (Info) IDs
Searching For Mephisto (bands/_/3540300069)                     True
Searching For Warskull (bands/_/3540400326)                     True
Searching For 

Searching For Buried Alive (bands/_/37681)                      True
Searching For Mob Rules (bands/_/712)                           True
Searching For Schatten (bands/_/3540447582)                     True
360/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 17 Minutes.
Saving 9522 MetalArchives Searched For Artist (Info) IDs
Searching For Neurosis (bands/_/13873)                          True
Searching For Hellbeyond (bands/_/3540269578)                   True
Searching For Born Addicted (bands/_/3540327235)                True
Searching For Dementhor (bands/_/3540429618)                    True
Searching For Saxorior (bands/_/8976)                           True
365/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 18 Minutes.
Saving 9527 MetalArchives Searched For Artist (Info) IDs
Searching For Afterbirth (bands/_/92553)                        True
Searching For Thrones of the Tyrant (bands/_/3540356925)        True
Searching Fo

Searching For Bloodfiend (bands/_/3540317770)                   True
Searching For Distorted Reality (bands/_/56344)                 True
Searching For Sumokem (bands/_/3540393760)                      True
Searching For Oracle Sun (bands/_/56128)                        True
Searching For Blood Freak (bands/_/13217)                       True
415/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 28 Minutes.
Saving 9577 MetalArchives Searched For Artist (Info) IDs
Searching For I Will Kill You (bands/_/3540335478)              True
Searching For Crime at the Morgue Street (bands/_/118508)       True
Searching For Funeral Nation (bands/_/12247)                    True
Searching For Midnight Messiah (bands/_/3540362196)             True
Searching For Avatar of Hate (bands/_/3540450185)               True
420/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 29 Minutes.
Saving 9582 MetalArchives Searched For Artist (Info) IDs
Searching Fo

Searching For Cruelty Force (bands/_/3540428641)                True
Searching For Hammer of the Gods (bands/_/3540323729)           True
Searching For Nuclear Wargod (bands/_/3540468963)               True
Searching For Abominated (bands/_/3540480296)                   True
Searching For Warfield (bands/_/3540280261)                     True
465/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 39 Minutes.
Saving 9627 MetalArchives Searched For Artist (Info) IDs
Searching For Unearthly Trance (bands/_/8520)                   True
Searching For Hybris (bands/_/43861)                            True
Searching For Bloody Tears (bands/_/31790)                      True
Searching For Apostasy (bands/_/3540262504)                     Error ==> Apostasy
Searching For Skullcap (bands/_/39082)                          True
Searching For Slowburn (bands/_/3540463035)                     True
470/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour an

In [7]:
#metalarchives.moveLocalFiles()